# The Physics Behind the Hangboard — Rohmert's Fatigue Curve

> *How long can you hold a given percentage of your maximum strength?*
> *And what does a 23-second hang really tell you about your MVC?*

This notebook explains **Rohmert's isometric endurance curve** — the
mathematical relationship between the intensity of a static muscle
contraction and how long it can be sustained. It is the foundation of
every hangboard calculator and the reason a 7-second test works at all.

**What you will learn:**

1. The original Rohmert equation and what each parameter means
2. How to plot force vs. time-to-failure
3. How to convert any (force, duration) pair to an MVC-7 equivalent
4. A practical worked example using the `climbing_science` library

## Background — Why This Curve Matters

In 1960 the German physiologist **Wolfgang Rohmert** published the first
systematic study of isometric muscle endurance (Rohmert 1960). He
measured how long factory workers could sustain static contractions at
different percentages of their maximum voluntary contraction (MVC).

The key finding: **the relationship between force and hold time is
non-linear and highly predictable.** At forces above ~15 % MVC, blood
flow is progressively occluded, metabolic by-products accumulate, and
the muscle fatigues on a well-defined curve.

Later refinements by **Sjøgaard (1986)** and **Frey-Law & Avin (2010)**
validated the shape of the curve and proposed parametric models that
fit a wide range of muscle groups and fitness levels.

**For climbers this means:**

- A short maximal hang (~7 s at 100 % MVC) and a longer sub-maximal
  hang (e.g. 23 s at 80 % MVC) carry *exactly the same information*
  about finger strength — once you know the curve.
- Every hangboard calculator (StrengthClimbing, Lattice, Climbro)
  relies on some variant of this curve.
- Understanding the curve lets you design your own protocols and
  interpret sub-maximal tests correctly.

## Setup

Import libraries and enable inline plotting.

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from climbing_science.strength import rohmert_conversion
from climbing_science.frontends.notebook import plot_rohmert_curve

## Your Data — Change Only This Cell

Enter your personal data below.  Every calculation in this notebook uses
these three variables.  You should not need to edit any other cell.

| Variable | Meaning | Unit |
|---|---|---|
| `MVC7_KG` | Your best 7-second max hang (added weight + bodyweight) | kg |
| `BODYWEIGHT_KG` | Your body weight | kg |
| `TEST_EDGE_MM` | Edge depth used for the test | mm |

In [ ]:
# ── YOUR DATA (edit here) ─────────────────────────────────────────────────
MVC7_KG = 105          # Maximum voluntary contraction for 7 s [kg]
BODYWEIGHT_KG = 72     # Body weight [kg]
TEST_EDGE_MM = 20      # Edge depth [mm]

# Derived quantity
ADDED_WEIGHT_KG = MVC7_KG - BODYWEIGHT_KG  # Weight added to harness [kg]
print(f"MVC-7: {MVC7_KG} kg  (BW {BODYWEIGHT_KG} kg + {ADDED_WEIGHT_KG} kg added)")
print(f"Test edge: {TEST_EDGE_MM} mm")

## The Rohmert Equation

Rohmert's endurance model predicts the **maximum hold time** $t_{\max}$
for a static contraction at relative force $f_{\text{rel}}$ (fraction of
MVC, range 0–1):

$$
t_{\max} = \left( -1.5 + \left(\frac{2.1}{f_{\text{rel}} - 0.15}\right)^{0.618} \right) \times 60 \;\;\text{[seconds]}
$$

| Symbol | Meaning | Range |
|---|---|---|
| $f_{\text{rel}}$ | Applied force / MVC (dimensionless) | (0.15, 1.0] |
| $t_{\max}$ | Time to failure | seconds |
| 0.15 | Asymptotic threshold — below ~15 % MVC the contraction can be sustained indefinitely (Rohmert 1960) | — |
| 2.1, 0.618 | Empirical curve-shape constants fitted to pooled data (Sjøgaard 1986; Frey-Law & Avin 2010) | — |
| 60 | Unit conversion from minutes to seconds | — |

**Key insight:** at $f_{\text{rel}} = 1.0$ (100 % MVC) the formula yields
$t_{\max} \approx 7\text{ s}$ — which is exactly why the standard
finger-strength test uses a **7-second hang** (Rohmert 1960).

In [ ]:
def rohmert_time_to_failure(f_rel: float) -> float:
    """Maximum hold time at a given fraction of MVC.

    Implements the Rohmert endurance model (Rohmert 1960) with constants
    from Sjøgaard (1986) and Frey-Law & Avin (2010).

    Args:
        f_rel: Relative force, fraction of MVC (0.15 < f_rel <= 1.0).

    Returns:
        Maximum hold time in seconds.
    """
    if f_rel <= 0.15 or f_rel > 1.0:
        return float("inf") if f_rel <= 0.15 else float("nan")
    return (-1.5 + (2.1 / (f_rel - 0.15)) ** 0.618) * 60

# Verify the canonical data point: 100 % MVC → ~7 s
t_at_100 = rohmert_time_to_failure(1.0)
print(f"At 100 % MVC: t_max = {t_at_100:.1f} s  (expected ≈ 7 s)")

### Plotting the Curve — Force vs. Time-to-Failure

Below we sweep $f_{\text{rel}}$ from 20 % to 100 % MVC and plot the
predicted time-to-failure.  The characteristic shape — a steep drop at
high intensities and a long tail at low intensities — is what makes
sub-maximal testing possible (Frey-Law & Avin 2010).

In [ ]:
# Sweep from 20 % to 100 % MVC
f_rel_values = np.linspace(0.20, 1.00, 200)
t_max_values = np.array([rohmert_time_to_failure(f) for f in f_rel_values])

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(f_rel_values * 100, t_max_values, linewidth=2.5, color="#2563EB")

# Mark the canonical 100 % MVC → 7 s data point
ax.plot(100, rohmert_time_to_failure(1.0), "o", color="#DC2626",
        markersize=10, zorder=5, label=f"100 % MVC → {rohmert_time_to_failure(1.0):.1f} s")

# Mark the 15 % MVC asymptote
ax.axvline(15, color="gray", linestyle="--", alpha=0.6, label="15 % MVC asymptote")

ax.set_xlabel("Force  [% MVC]", fontsize=13)
ax.set_ylabel("Time to failure  [s]", fontsize=13)
ax.set_title("Rohmert's Isometric Endurance Curve  (Rohmert 1960)", fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(15, 105)
ax.set_ylim(0, max(t_max_values) * 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Interactive Exploration — What Happens at Different Intensities?

The table below shows time-to-failure for selected %MVC values.
Notice how the relationship is strongly non-linear: dropping from 90 %
to 80 % MVC roughly doubles the hold time, while dropping from 30 %
to 20 % adds *minutes* (Rohmert 1960; Sjøgaard 1986).

In [ ]:
# Time-to-failure for selected %MVC intensities
pct_mvc_levels = [20, 30, 40, 50, 60, 70, 80, 90, 95, 100]

print(f"{'%MVC':>6}  {'t_max (s)':>10}  {'t_max (min)':>11}  {'Bar'}")
print("-" * 55)
for pct in pct_mvc_levels:
    t = rohmert_time_to_failure(pct / 100)
    bar = "█" * min(int(t / 10), 60)  # Scale: 1 block = 10 s
    print(f"{pct:>5}%  {t:>10.1f}  {t/60:>10.1f}m  {bar}")

print()
print("Key observations:")
print(f"  • At 100 % MVC → {rohmert_time_to_failure(1.0):.1f} s  (the 7-s test)")
print(f"  • At  50 % MVC → {rohmert_time_to_failure(0.50):.1f} s  (~1 min)")
print(f"  • At  20 % MVC → {rohmert_time_to_failure(0.20):.1f} s  (many minutes)")

In [ ]:
# Detailed sweep: %MVC from 20 % to 95 % in fine steps
f_range = np.linspace(0.20, 0.95, 300)
t_range = np.array([rohmert_time_to_failure(f) for f in f_range])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left panel — linear scale
ax1.plot(f_range * 100, t_range, linewidth=2, color="#2563EB")
ax1.fill_between(f_range * 100, t_range, alpha=0.08, color="#2563EB")
ax1.set_xlabel("Force  [% MVC]", fontsize=12)
ax1.set_ylabel("Time to failure  [s]", fontsize=12)
ax1.set_title("Linear scale", fontsize=13)
ax1.grid(True, alpha=0.3)

# Right panel — log scale on y-axis (reveals the power-law shape)
ax2.semilogy(f_range * 100, t_range, linewidth=2, color="#DC2626")
ax2.set_xlabel("Force  [% MVC]", fontsize=12)
ax2.set_ylabel("Time to failure  [s]  (log scale)", fontsize=12)
ax2.set_title("Log scale — reveals the power-law shape", fontsize=13)
ax2.grid(True, alpha=0.3, which="both")

fig.suptitle("Rohmert curve: 20–95 % MVC  (Rohmert 1960; Frey-Law & Avin 2010)",
             fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## Library Plot — Multiple Fitness Models

The `climbing_science` library ships three parametric Rohmert models —
*beginner*, *intermediate*, and *expert* — reflecting that trained
athletes can sustain a given %MVC for longer than untrained subjects
(Frey-Law & Avin 2010).

Below we use the built-in `plot_rohmert_curve()` helper.

In [ ]:
# Plot all three fitness models using the library helper
fig = plot_rohmert_curve(
    models=["beginner", "intermediate", "expert"],
    figsize=(10, 6),
    show=True,
)

## Practical Example — From a Sub-Maximal Hang to MVC-7

**Scenario:** You cannot (or prefer not to) do a true maximal 7-second
hang.  Instead you hang with +20 kg of added weight at a body weight
of 72 kg on a 20 mm edge and hold for **23 seconds** before failure.

**Question:** What is your equivalent MVC-7 (the load you could hold
for exactly 7 seconds)?

We use `rohmert_conversion()` from `climbing_science.strength`.
The function finds the %MVC that produces a 23-second failure time
on the Rohmert curve, then scales it to the 7-second equivalent.

In [ ]:
# ── Sub-maximal test data ─────────────────────────────────────────────────
ADDED_WEIGHT_TEST_KG = 20    # Weight added during the test [kg]
HANG_DURATION_SEC = 23       # Time held before failure [s]

# Total load during the test
test_load_kg = BODYWEIGHT_KG + ADDED_WEIGHT_TEST_KG
print(f"Test load: {BODYWEIGHT_KG} kg BW + {ADDED_WEIGHT_TEST_KG} kg added = {test_load_kg} kg")
print(f"Hang duration: {HANG_DURATION_SEC} s")

# The test load as a fraction of (unknown) true MVC is what we need.
# rohmert_conversion expects force_percent_mvc and duration, and returns
# the equivalent %MVC at the target duration (default 7 s).
#
# Strategy: we know the duration (23 s) tells us the %MVC on the curve.
# The Rohmert equation gives: 23 s ↔ some f_rel.
# Then MVC_true = test_load / f_rel, and MVC-7 = MVC_true × f_rel_at_7s.

# Find the %MVC that corresponds to a 23-second hold
from scipy.optimize import brentq

def _residual(f_rel):
    return rohmert_time_to_failure(f_rel) - HANG_DURATION_SEC

# Solve: we know 23 s is between 100 % (~7 s) and 20 % (~several min)
f_rel_test = brentq(_residual, 0.16, 1.0)
print(f"\nRohmert says {HANG_DURATION_SEC} s corresponds to {f_rel_test:.1%} MVC")

# True MVC = test_load / f_rel
mvc_true_kg = test_load_kg / f_rel_test
print(f"Estimated true MVC (instantaneous): {mvc_true_kg:.1f} kg")

# MVC-7 = true MVC × f_rel at 7 s
f_rel_at_7s = brentq(lambda f: rohmert_time_to_failure(f) - 7.0, 0.16, 1.0)
mvc7_estimated_kg = mvc_true_kg * f_rel_at_7s
print(f"f_rel at 7 s: {f_rel_at_7s:.3f}")
print(f"\n═══════════════════════════════════════════")
print(f"  Estimated MVC-7: {mvc7_estimated_kg:.1f} kg")
print(f"  (BW {BODYWEIGHT_KG} kg + {mvc7_estimated_kg - BODYWEIGHT_KG:.1f} kg added)")
print(f"═══════════════════════════════════════════")

### Shortcut — Using `rohmert_conversion()` Directly

The library function `rohmert_conversion` does the same calculation in
one call.  You provide the force as a **percentage of MVC** and the
duration, and it returns the equivalent percentage at the target
duration (default: 7 s).

Since we don't know the true MVC, we express the test load as a
percentage of a *hypothetical* MVC and solve for the one that is
self-consistent.

In [ ]:
# Using the library's rohmert_conversion directly
# If we assume the test load IS our MVC (100 %), and we held for 23 s,
# the function tells us what %MVC-at-7s that corresponds to.
equiv_pct = rohmert_conversion(
    force_percent_mvc=100.0,  # Assume test load = 100 % MVC
    duration_sec=HANG_DURATION_SEC,
    target_duration_sec=7.0,
)
print(f"rohmert_conversion(100%, {HANG_DURATION_SEC}s, target=7s) = {equiv_pct:.1f}%")
print(f"This means the test load ({test_load_kg} kg) is {equiv_pct:.1f}% of MVC-7.")
print()

# Therefore MVC-7 = test_load / (equiv_pct / 100)
mvc7_from_lib = test_load_kg / (equiv_pct / 100.0)
print(f"MVC-7 = {test_load_kg} kg / {equiv_pct:.1f}% = {mvc7_from_lib:.1f} kg")
print(f"Added weight at MVC-7: {mvc7_from_lib - BODYWEIGHT_KG:.1f} kg")

### Visualising the Conversion on the Curve

The plot below shows where the sub-maximal test sits on the Rohmert
curve and where the 7-second equivalent lands.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the curve
f_sweep = np.linspace(0.20, 1.00, 300)
t_sweep = np.array([rohmert_time_to_failure(f) for f in f_sweep])
ax.plot(f_sweep * 100, t_sweep, linewidth=2.5, color="#2563EB", label="Rohmert curve")

# Mark the sub-maximal test point
ax.plot(f_rel_test * 100, HANG_DURATION_SEC, "s", color="#F59E0B",
        markersize=14, zorder=5,
        label=f"Your test: {f_rel_test:.0%} MVC, {HANG_DURATION_SEC} s")

# Mark the MVC-7 point
ax.plot(f_rel_at_7s * 100, 7.0, "D", color="#DC2626",
        markersize=12, zorder=5,
        label=f"MVC-7 equivalent: {f_rel_at_7s:.0%} MVC, 7 s")

# Draw connecting arrows
ax.annotate("", xy=(f_rel_at_7s * 100, 7.0),
            xytext=(f_rel_test * 100, HANG_DURATION_SEC),
            arrowprops=dict(arrowstyle="->", color="gray", lw=1.5, ls="--"))

ax.set_xlabel("Force  [% MVC]", fontsize=13)
ax.set_ylabel("Time to failure  [s]", fontsize=13)
ax.set_title("Sub-maximal test → MVC-7 conversion  (Rohmert 1960)", fontsize=14)
ax.legend(fontsize=11, loc="upper right")
ax.set_xlim(15, 105)
ax.set_ylim(0, max(t_sweep) * 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Key Takeaways

1. **Rohmert's curve** describes a universal, non-linear relationship
   between static force and endurance time (Rohmert 1960).

2. **Below ~15 % MVC** the contraction can be sustained indefinitely —
   blood flow is sufficient to clear metabolites (Sjøgaard 1986).

3. **At 100 % MVC** the predicted hold time is **≈ 7 seconds**, which
   is exactly why the standard finger-strength test uses a 7 s duration.

4. **Sub-maximal tests are valid:** a longer hang at lower intensity
   carries the same information as a short maximal hang, once converted
   through the Rohmert curve (Frey-Law & Avin 2010).

5. **Practical rule of thumb:** if you can hold a load for ~23 s, that
   load is roughly 75–80 % of your MVC-7, depending on the model used.

6. **Fitness level matters:** trained climbers ("expert" model) sustain
   a given %MVC longer than beginners due to better capillarisation,
   fibre-type distribution, and motor-unit recruitment efficiency
   (Frey-Law & Avin 2010).

## References

1. **Rohmert W.** (1960). *Ermittlung von Erholungspausen für statische
   Arbeit des Menschen.* Internationale Zeitschrift für angewandte
   Physiologie einschließlich Arbeitsphysiologie, 18, 123–164.
   — Original isometric endurance model.

2. **Sjøgaard G.** (1986). *Intramuscular changes during long-term
   contraction.* In N. L. Jones, N. McCartney & A. J. McComas (Eds.),
   *Human Muscle Power* (pp. 113–128). Human Kinetics.
   — Physiological mechanisms behind the Rohmert curve.

3. **Frey-Law L. A. & Avin K. G.** (2010). *Endurance time is
   joint-specific: A modelling and meta-analysis investigation.*
   Ergonomics, 53(1), 109–129.
   DOI: [10.1080/00140130903389068](https://doi.org/10.1080/00140130903389068)
   — Parametric curve fitting across joints and fitness levels.

4. **Giles D., Romero V., Lechner R.** (2019). *Lattice Training
   finger-strength benchmarks.* Internal report, Lattice Training Ltd.
   — Context for MVC-7 benchmarking in climbing.

---

*Notebook created for the
[climbing-science](https://github.com/gerolfziegenhain/climbing-science)
library.*